# 灰度图像形态学实验：膨胀、腐蚀、开运算与闭运算（学生练习版）

本 Notebook 是学生练习版：网络灰度图像读取、显示和对比实验已经保留；灰度形态学核心算法被改为 `TODO`，需要学生补全。

本实验面向 **灰度图像**，实现并观察四种常见形态学操作：

1. **灰度膨胀 Dilation**
2. **灰度腐蚀 Erosion**
3. **灰度开运算 Opening**
4. **灰度闭运算 Closing**

与二值图像不同，灰度图像的像素值不是只有 `0` 和 `1`，而是通常位于 `0~255` 范围内。灰度形态学操作不再只判断前景和背景，而是通过局部最大值或局部最小值改变图像的亮暗结构。

## 1. 相关背景知识

### 1.1 灰度图像

灰度图像中，每个像素都有一个灰度值：

- `0` 表示黑色。
- `255` 表示白色。
- 中间值表示不同亮度的灰色。

在灰度形态学中，亮区域和暗区域都会受到结构元素的影响。

### 1.2 结构元素

结构元素可以理解为一个小窗口。本实验使用方形结构元素，例如：

- `3 × 3`
- `5 × 5`
- `9 × 9`

结构元素越大，参与局部最大值或最小值计算的邻域越大，图像变化也越明显。

### 1.3 灰度膨胀

灰度膨胀会把当前像素替换为邻域中的 **最大灰度值**。

直观效果：

- 亮区域会向周围扩张。
- 小暗点可能被周围亮像素填掉。
- 图像整体亮结构会变粗、变大。

### 1.4 灰度腐蚀

灰度腐蚀会把当前像素替换为邻域中的 **最小灰度值**。

直观效果：

- 暗区域会向周围扩张。
- 小亮点可能被周围暗像素去掉。
- 图像整体亮结构会变细、变小。

### 1.5 灰度开运算

灰度开运算定义为：

`开运算 = 先腐蚀，再膨胀`

常见效果：

- 去除小的亮噪声或亮细节。
- 保留较大的暗背景结构。
- 平滑亮目标边界。

### 1.6 灰度闭运算

灰度闭运算定义为：

`闭运算 = 先膨胀，再腐蚀`

常见效果：

- 填补小的暗孔洞或暗裂缝。
- 保留较大的亮目标结构。
- 平滑暗缺陷造成的边界。

## 2. 实验图像来源

本实验使用一张来自网络的真实图像作为灰度形态学实验对象。程序会完成以下步骤：

1. 从 Wikimedia Commons 下载一张公开图像。
2. 将彩色图像转换为灰度图像。
3. 将图像缩放到适合课堂运行的大小。
4. 在灰度图像上执行膨胀、腐蚀、开运算和闭运算。

使用真实图像可以更直观地观察灰度形态学对亮区域、暗区域、纹理细节和局部噪声的影响。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from io import BytesIO
from urllib.request import Request, urlopen
from pathlib import Path
import shutil
import subprocess
import tempfile

np.random.seed(42)

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False


def show_gray_image(image, title="", ax=None, vmin=0, vmax=255):
    """显示单张灰度图像。

    参数：
    - image：二维 NumPy 数组，表示灰度图像。
    - title：图像标题。
    - ax：Matplotlib 坐标轴对象；如果为 None，则创建新图。
    - vmin, vmax：显示灰度范围。
    """
    if ax is None:
        _, ax = plt.subplots(figsize=(4, 4))
    im = ax.imshow(image, cmap="gray", vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.axis("off")
    return im


def show_image_grid(images, titles, main_title="", cols=5, figsize=(14, 8), vmin=0, vmax=255):
    """以网格方式显示多张灰度图像。"""
    rows = int(np.ceil(len(images) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.array(axes).reshape(-1)
    for ax, image, title in zip(axes, images, titles):
        show_gray_image(image, title, ax, vmin=vmin, vmax=vmax)
    for ax in axes[len(images):]:
        ax.axis("off")
    fig.suptitle(main_title, fontsize=16)
    plt.tight_layout()
    plt.show()

In [ ]:
IMAGE_URL = "https://raw.githubusercontent.com/scikit-image/scikit-image/v0.25.2/skimage/data/camera.png"


def download_image_bytes(url: str) -> bytes:
    """从网络下载图像字节。

    优先使用 Python 标准库下载；如果当前环境 HTTPS 连接失败，则尝试调用系统 curl。
    """
    try:
        request = Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urlopen(request, timeout=20) as response:
            return response.read()
    except Exception as python_error:
        curl_path = shutil.which("curl.exe") or shutil.which("curl")
        if curl_path is None:
            raise RuntimeError(f"Python 下载失败，且未找到 curl。原始错误：{python_error}") from python_error

        with tempfile.NamedTemporaryFile(delete=False, suffix=".png") as tmp:
            tmp_path = tmp.name

        command = [
            curl_path,
            "-L",
            "-A",
            "Mozilla/5.0",
            "-o",
            tmp_path,
            url,
        ]
        if curl_path.lower().endswith("curl.exe"):
            command.insert(1, "--ssl-no-revoke")

        completed = subprocess.run(command, capture_output=True, text=True)
        if completed.returncode != 0:
            raise RuntimeError(
                "图像下载失败，请检查网络连接。\n"
                f"Python 错误：{python_error}\n"
                f"curl 错误：{completed.stderr}"
            ) from python_error

        data = Path(tmp_path).read_bytes()
        Path(tmp_path).unlink(missing_ok=True)
        return data


def load_network_image(url: str) -> np.ndarray:
    """从网络读取图像。

    参数：
    - url：图像文件的网络地址。

    返回：
    - NumPy 图像数组。彩色图像通常形状为 `(height, width, 3)`。
    """
    data = download_image_bytes(url)
    return plt.imread(BytesIO(data), format="png")


def rgb_to_grayscale(image: np.ndarray) -> np.ndarray:
    """将彩色图像转换为灰度图像。

    参数：
    - image：彩色或灰度图像数组。

    返回：
    - 0~255 范围内的二维灰度图像。
    """
    if image.ndim == 2:
        gray = image.astype(float)
    else:
        rgb = image[..., :3].astype(float)
        gray = 0.299 * rgb[..., 0] + 0.587 * rgb[..., 1] + 0.114 * rgb[..., 2]

    if gray.max() <= 1.0:
        gray = gray * 255.0
    return np.clip(gray, 0, 255).astype(np.uint8)


def resize_by_sampling(image: np.ndarray, max_side: int = 180) -> np.ndarray:
    """用简单采样方式缩小图像，避免形态学循环运行过慢。

    参数：
    - image：输入灰度图像。
    - max_side：缩放后最长边不超过该值。

    返回：
    - 缩小后的灰度图像。
    """
    h, w = image.shape
    step = max(1, int(np.ceil(max(h, w) / max_side)))
    return image[::step, ::step]


network_image = load_network_image(IMAGE_URL)
gray_image = resize_by_sampling(rgb_to_grayscale(network_image), max_side=180)

print(f"网络图像地址：{IMAGE_URL}")
print(f"原始图像形状：{network_image.shape}")
print(f"灰度实验图像形状：{gray_image.shape}")
show_gray_image(gray_image, "网络图像转换后的灰度实验图像")
plt.show()

## 3. 灰度形态学运算程序

下面从零实现灰度膨胀、灰度腐蚀、灰度开运算和灰度闭运算。为了便于观察原理，本实验直接使用 `numpy` 编写局部最大值和局部最小值操作，不调用 OpenCV。

本实验使用 **方形结构元素**。如果结构元素大小为 `5`，则表示每个像素会参考其周围 `5 × 5` 邻域。

### 3.1 算法补全步骤

请按下面顺序完成灰度形态学算法：

1. 补全 `create_square_structuring_element(size)`，生成方形结构元素。
2. 补全 `grayscale_dilation(image, se)`，实现灰度膨胀：输出邻域最大灰度值。
3. 补全 `grayscale_erosion(image, se)`，实现灰度腐蚀：输出邻域最小灰度值。
4. 补全 `grayscale_opening(image, se)`，按“先腐蚀，再膨胀”实现开运算。
5. 补全 `grayscale_closing(image, se)`，按“先膨胀，再腐蚀”实现闭运算。
6. 运行后续单元，比较不同结构元素大小下的图像变化。

In [ ]:
def create_square_structuring_element(size=3):
    """创建方形结构元素。"""
    # TODO 1：判断 size 是否为奇数。
    # TODO 2：返回一个大小为 size × size 的布尔数组。
    raise NotImplementedError("请补全 create_square_structuring_element 函数")


def grayscale_dilation(image, se):
    """灰度膨胀：取结构元素覆盖区域内的最大灰度值。"""
    # TODO 1：根据结构元素大小对 image 做边界填充，建议使用 mode="edge"。
    # TODO 2：遍历每个像素，取出邻域 region。
    # TODO 3：计算 region 中 se 为 True 的位置的最大值，作为输出像素。
    # TODO 4：返回膨胀后的灰度图像。
    raise NotImplementedError("请补全 grayscale_dilation 函数")


def grayscale_erosion(image, se):
    """灰度腐蚀：取结构元素覆盖区域内的最小灰度值。"""
    # TODO 1：根据结构元素大小对 image 做边界填充，建议使用 mode="edge"。
    # TODO 2：遍历每个像素，取出邻域 region。
    # TODO 3：计算 region 中 se 为 True 的位置的最小值，作为输出像素。
    # TODO 4：返回腐蚀后的灰度图像。
    raise NotImplementedError("请补全 grayscale_erosion 函数")


def grayscale_opening(image, se):
    """灰度开运算：先腐蚀，再膨胀。"""
    # TODO：调用 grayscale_erosion 和 grayscale_dilation 实现开运算。
    raise NotImplementedError("请补全 grayscale_opening 函数")


def grayscale_closing(image, se):
    """灰度闭运算：先膨胀，再腐蚀。"""
    # TODO：调用 grayscale_dilation 和 grayscale_erosion 实现闭运算。
    raise NotImplementedError("请补全 grayscale_closing 函数")

## 4. 基本灰度形态学操作对比

下面统一使用 `5 × 5` 方形结构元素，对同一幅灰度图像执行：

- 原图
- 灰度膨胀
- 灰度腐蚀
- 灰度开运算
- 灰度闭运算

In [ ]:
def compare_basic_grayscale_operations(image, se_size=5):
    """展示灰度图像四种基本形态学操作。"""
    se = create_square_structuring_element(se_size)
    images = [
        image,
        grayscale_dilation(image, se),
        grayscale_erosion(image, se),
        grayscale_opening(image, se),
        grayscale_closing(image, se),
    ]
    titles = ["原图", "灰度膨胀", "灰度腐蚀", "灰度开运算", "灰度闭运算"]
    show_image_grid(
        images,
        titles,
        main_title=f"灰度形态学基本操作对比：方形结构元素 {se_size}×{se_size}",
        cols=5,
        figsize=(15, 3.5),
    )


compare_basic_grayscale_operations(gray_image, se_size=5)

## 5. 不同结构元素大小的影响

结构元素越大，局部最大值和最小值计算的邻域越大。

因此：

- 灰度膨胀会让亮区域扩张得更明显。
- 灰度腐蚀会让暗区域扩张得更明显。
- 灰度开运算会更强烈地去除小亮点。
- 灰度闭运算会更强烈地填补小暗点或暗裂缝。

下面比较 `3 × 3`、`5 × 5`、`9 × 9` 三种结构元素大小。

In [ ]:
def compare_sizes(image, operation_name, operation_func, sizes=(3, 5, 9)):
    """比较不同结构元素大小对同一灰度形态学操作的影响。"""
    images = [image]
    titles = ["原图"]
    for size in sizes:
        se = create_square_structuring_element(size)
        images.append(operation_func(image, se))
        titles.append(f"{size}×{size}")
    show_image_grid(
        images,
        titles,
        main_title=f"{operation_name}：不同结构元素大小的影响",
        cols=len(images),
        figsize=(14, 3.5),
    )


compare_sizes(gray_image, "灰度膨胀", grayscale_dilation)
compare_sizes(gray_image, "灰度腐蚀", grayscale_erosion)
compare_sizes(gray_image, "灰度开运算", grayscale_opening)
compare_sizes(gray_image, "灰度闭运算", grayscale_closing)

## 6. 开运算与闭运算的细节观察

为了更清楚地观察开运算和闭运算的作用，可以计算操作前后的差值图：

- `原图 - 开运算结果`：可以突出被开运算去除的亮细节。
- `闭运算结果 - 原图`：可以突出被闭运算填补的暗细节。

In [ ]:
def show_open_close_details(image, se_size=5):
    """显示开运算和闭运算对亮细节、暗细节的影响。"""
    se = create_square_structuring_element(se_size)
    opened = grayscale_opening(image, se)
    closed = grayscale_closing(image, se)

    bright_removed = np.clip(image.astype(int) - opened.astype(int), 0, 255).astype(np.uint8)
    dark_filled = np.clip(closed.astype(int) - image.astype(int), 0, 255).astype(np.uint8)

    show_image_grid(
        [image, opened, bright_removed, closed, dark_filled],
        ["原图", "开运算", "被开运算去除的亮细节", "闭运算", "被闭运算填补的暗细节"],
        main_title=f"开运算与闭运算细节观察：方形结构元素 {se_size}×{se_size}",
        cols=5,
        figsize=(16, 3.5),
    )


show_open_close_details(gray_image, se_size=5)

## 7. 实验思考

完成实验后，可以思考下面的问题：

1. 灰度膨胀为什么会让亮区域扩张？
2. 灰度腐蚀为什么会让暗区域扩张？
3. 灰度开运算为什么适合去除小亮点？
4. 灰度闭运算为什么适合填补小暗点或暗裂缝？
5. 当结构元素从 `3 × 3` 增大到 `9 × 9` 时，图像变化为什么更明显？
6. 二值形态学和灰度形态学在运算逻辑上有什么相同点和不同点？